In [32]:
import os
from dotenv import load_dotenv
from langchain_cerebras import ChatCerebras
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langchain.agents import create_agent
from pydantic import BaseModel
import sib_api_v3_sdk
from sib_api_v3_sdk.rest import ApiException


load_dotenv()
os.environ["CEREBRAS_API_KEY"] = os.getenv("CEREBRAS_API_KEY")
os.environ["GROQ_API_KEY"]     = os.getenv("GROQ_API_KEY")
BREVO_KEY = os.getenv("BREVO_API_KEY")
MONGODB_URI=os.getenv("MONGODB_URI")
llm_gpt = ChatCerebras(
    model="gpt-oss-120b",
    api_key=os.getenv("CEREBRAS_API_KEY"),
)
llm_groq = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY"),
)


In [33]:
class User(BaseModel):
    id: int
    name: str
    email: str
    minutes_listened: int
    country: str
    joined_at: str


class EmailDraft(BaseModel):
    recipient: str
    subject: str
    body: str
    reason: str


class CampaignResult(BaseModel):
    total_users: int
    emails_sent: int
    failed: int


In [34]:
from pymongo import MongoClient

client = MongoClient(MONGODB_URI)

db = client["soulsync"]

users_collection = db["users"]



In [35]:
@tool
def GetTopUsers(limit: int):
    """
    Return the top users ranked by listening time (in minutes).
    """
    top_users = list(
        users_collection.find(
            {},
            {
                "_id": 0,
                "name": 1,
                "email": 1,
                "totalListeningTime": 1,
            }
        )
        .sort("totalListeningTime", -1)
        .limit(limit)
    )

    for user in top_users:
        user["minutesListened"] = round(user["totalListeningTime"] / 60)
        del user["totalListeningTime"]

    return top_users

In [36]:
@tool
def sendMail(
    sender: str,
    receiver: str,
    subject: str,
    body: str,
) -> str:
    """Send an email using Brevo."""

    configuration = sib_api_v3_sdk.Configuration()
    configuration.api_key["api-key"] = BREVO_KEY

    api_instance = sib_api_v3_sdk.TransactionalEmailsApi(
        sib_api_v3_sdk.ApiClient(configuration)
    )

    send_smtp_email = sib_api_v3_sdk.SendSmtpEmail(
        sender={"name": "SoulSync", "email": "lokeshvijayraina@gmail.com"},
        to=[{"email": receiver}],
        subject=subject,
        html_content=body,
    )

    try:
        api_response = api_instance.send_transac_email(send_smtp_email)
        print("Success! Message ID:", api_response.message_id)
        print(sender, receiver, subject)
        return f"Mail sent successfully to {receiver}"
    except ApiException as e:
        print(" Error:", e)
        return f"Failed to send email: {e}"


In [38]:
agent = create_agent(
    model=llm_gpt,
    tools=[GetTopUsers, sendMail],
    system_prompt="""
You are LOOKOUT, an email assistant working on behalf of SoulSync, a music streaming platform.

You have two tools:

1. GetTopUsers(limit) – returns the highest-ranked users.
2. sendMail(sender, receiver, subject, body) – sends an email.

Workflow:
1. Find the top users.
2. Write a professional, warm thank-you email for each, on behalf of SoulSync.
3. Send each email.
4. Repeat until every user has received one.

Tone and style:
- Professional but warm, like a real brand writing to a valued customer.
- Avoid generic filler phrases ("fuels our community", "excited to bring you fresh tracks").
- Only use numbers explicitly provided by GetTopUsers. Never invent additional statistics or totals.
- Sign off as "The SoulSync Team".
"""
)

response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="""
Reward our top 1 users.
Write a professional thank-you email for each user and send it.
"""
            )
        ]
    }
)

Success! Message ID: <202607051825.19960732803@smtp-relay.mailin.fr>
no-reply@soulsync.com lokeshvijayraina@gmail.com Thank You for Being a Top Listener
